# Artwork Eval on Colab (No Drive)

This notebook runs the artwork benchmark directly in Colab using git clones under `/content`.

- Benchmark repo: `physical168/kv-compression-benchmark`
- Engine repo: `GabrieleSanmartino/CompressionExperiments`
- Entry point: `CompressionExperiments/experiment_manager/run_compression_image.py`

> No Google Drive is required.

## Step 0 - Runtime

Set runtime to **GPU** in Colab before running.

## Step 0.5 - GitHub Classic PAT input

Use a **Classic Personal Access Token** (scope: `repo`) for private repo clone.

You can either:
- type token in the input cell below (hidden), or
- leave empty and rely on Colab Secret `GITHUB_TOKEN`.

Notes:
- Public repos do not require token.
- Private repo clone will fail if token is missing/invalid.

In [ ]:
import getpass
import os

# 1) manual hidden input (preferred when you don't want to use Colab Secrets)
manual = getpass.getpass("Paste GitHub Classic PAT (leave empty to use Colab Secret GITHUB_TOKEN): ").strip()

# 2) fallback to Colab Secret if manual input is empty
if manual:
    gh_token = manual
else:
    try:
        from google.colab import userdata
        gh_token = (userdata.get("GITHUB_TOKEN") or "").strip()
    except Exception:
        gh_token = os.environ.get("GITHUB_TOKEN", "").strip()

print("Token loaded:", bool(gh_token))

In [ ]:
from pathlib import Path
import os, shutil, subprocess

BENCH_URL = "https://github.com/physical168/kv-compression-benchmark.git"
CE_URL = "https://github.com/GabrieleSanmartino/CompressionExperiments.git"

BENCH_DIR = Path("/content/kv-compression-benchmark")
CE_DIR = Path("/content/CompressionExperiments")


def clone_public_or_token(url: str, out_dir: Path, token: str | None) -> None:
    # Try public clone first.
    r = subprocess.run(["git", "clone", "--depth", "1", url, str(out_dir)], capture_output=True, text=True)
    if r.returncode == 0:
        return

    # If public clone failed, require token for private repo.
    if not token:
        raise RuntimeError(
            "Anonymous clone failed and GITHUB_TOKEN is missing. "
            "Set Colab Secret GITHUB_TOKEN and rerun this cell."
        )

    private_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    subprocess.check_call(["git", "clone", "--depth", "1", private_url, str(out_dir)])


gh_token = (globals().get("gh_token") or "").strip() or None

for p in [BENCH_DIR, CE_DIR]:
    if p.exists():
        shutil.rmtree(p)

subprocess.check_call(["git", "clone", "--depth", "1", BENCH_URL, str(BENCH_DIR)])
clone_public_or_token(CE_URL, CE_DIR, gh_token)
print("Cloned:", BENCH_DIR)
print("Cloned:", CE_DIR)

In [ ]:
%cd /content/CompressionExperiments
!git submodule update --init --recursive

!pip install -q -U pip
!pip install -q transformers accelerate bitsandbytes pandas pyyaml scikit-learn matplotlib tqdm pillow
!pip install -q -e kvpress

## Step 1 - Prepare artwork dataset

This step prepares `CompressionExperiments/experiment_manager/datasets/artwork/`.

Behavior:
- CSV: prefer benchmark repo `paintings.csv`; fallback to CE built-in `paintings.csv`.
- Images: prefer CE built-in `images/`; fallback to benchmark repo `images/`.

So if CE already contains images, no extra upload is needed.

In [ ]:
from pathlib import Path
import shutil

# Benchmark repo (optional source)
SRC_BENCH_ART = Path("/content/kv-compression-benchmark/datasets/artwork")
SRC_BENCH_CSV = SRC_BENCH_ART / "paintings.csv"
SRC_BENCH_IMG = SRC_BENCH_ART / "images"

# CE dataset destination (and possible built-in source)
DST_ART = Path("/content/CompressionExperiments/experiment_manager/datasets/artwork")
DST_ART.mkdir(parents=True, exist_ok=True)

# 1) CSV: prefer benchmark CSV, else keep CE CSV if present
if SRC_BENCH_CSV.is_file():
    shutil.copy2(SRC_BENCH_CSV, DST_ART / "paintings.csv")
    print("CSV source: benchmark repo")
elif (DST_ART / "paintings.csv").is_file():
    print("CSV source: CE built-in")
else:
    raise FileNotFoundError("No paintings.csv found in benchmark repo or CE dataset path.")

# 2) Images: prefer CE built-in images; fallback to benchmark images copy
ce_images = DST_ART / "images"
if ce_images.is_dir() and any(ce_images.iterdir()):
    print("Images source: CE built-in", ce_images)
elif SRC_BENCH_IMG.is_dir() and any(SRC_BENCH_IMG.iterdir()):
    if ce_images.exists():
        shutil.rmtree(ce_images)
    shutil.copytree(SRC_BENCH_IMG, ce_images)
    print("Images source: benchmark repo", SRC_BENCH_IMG)
else:
    raise FileNotFoundError(
        "No usable images found in CE dataset or benchmark repo dataset."
    )

print("Dataset ready:", DST_ART)
print("paintings.csv exists:", (DST_ART / "paintings.csv").is_file())
print("images dir exists:", ce_images.is_dir())
print("image count:", len(list(ce_images.glob("*"))))

## Step 2 - Run compression (image)

Default uses `ExpectedAttentionPress` only. Add other presses if needed.

In [ ]:
%cd /content/CompressionExperiments/experiment_manager

!python run_compression_image.py \
  --model llava-hf/llama3-llava-next-8b-hf \
  --press ExpectedAttentionPress \
  --ratios 0.4 0.8 0.95 \
  --max-records 20 \
  --output-dir results \
  --attn-impl sdpa

## Step 3 - Evaluate (P/R/F1)

In [ ]:
%cd /content/CompressionExperiments/experiment_manager
!python evaluate.py --results-dir results --summary

## Step 4 - Optional: Zip results for download

In [ ]:
%cd /content/CompressionExperiments/experiment_manager
!zip -r /content/ce_artwork_results.zip results
print("Created: /content/ce_artwork_results.zip")